# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahsan-Qadeer/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

warehouse = "hf://datasets/FlyRank/internship-warehouse"

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{token}')")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. My rule and its reason codes

Pages that have low clicks but large number of appearances (impressions) would be great for review since making them better could improve traffic. I have also used page views as a signal so that low traffic pages are not prioritized over higher impact pages.

###Reasons codes:
LOW_CTR

HIGH_IMPRESSIONS

###Action:
REVIEW_CONTENT



In [2]:
bins = [0,100,500,1000,5000,1000000]

df["impression_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=bins
)

table = (
    df.groupby("impression_bucket")
      .size()
      .reset_index(name="n")
)

print(table)

  impression_bucket        n
0          (0, 100]  2977578
1        (100, 500]   532347
2       (500, 1000]    68776
3      (1000, 5000]    31617
4   (5000, 1000000]      743


/tmp/ipykernel_6360/3624777921.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("impression_bucket")


The impression counts are in several buckets. This implies that the signal has enough variation to allow a basis for page review prioritization

In [3]:
bins = [0,10,50,100,500,1000000]

df["pageview_bucket"] = pd.cut(
    df["ga4_pageviews"],
    bins=bins
)

table2 = (
    df.groupby("pageview_bucket")
      .size()
      .reset_index(name="n")
)

print(table2)

  pageview_bucket       n
0         (0, 10]  387988
1        (10, 50]   23790
2       (50, 100]    1221
3      (100, 500]     310
4  (500, 1000000]       8


/tmp/ipykernel_6360/1075498971.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("pageview_bucket")


Page views also show alot of variation so it is useful for review prioritzation

## 2. Build the ranked queue (writes the CSV)

In the ranked queue, pages are ranked based on a simple score using search visbility and user engagement. Higher impression pages are nearer to the queue start while pages with lesser clicks are prioritised over pages that already have many clicks. The output is one action label REVIEW_CONTENT and one reason code HIGH_IMPRESSIONS_LOW_CLICKS.

In [4]:
import os

# Prevent divide-by-zero
df["gsc_clicks"] = df["gsc_clicks"].fillna(0)
df["gsc_impressions"] = df["gsc_impressions"].fillna(0)

# Simple baseline score
df["baseline_score"] = (
    df["gsc_impressions"] -
    (5 * df["gsc_clicks"])
)

# Reason code
df["reason_code"] = "HIGH_IMPRESSIONS_LOW_CLICKS"

# Action label
df["action"] = "REVIEW_CONTENT"

# Rank pages
queue = (
    df.sort_values("baseline_score", ascending=False)
      .reset_index(drop=True)
)

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Save CSV
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(queue[[
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action"
]].head(10))

print("\nCSV written successfully.")

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.